# Groups and memberships

Typed buckets of images. A `GroupType` defines the allowed and required roles. The
OpenSearch denorm carries `group_ids = ['<type>::<uuid>']` and
`memberships = ['<role>::<type>::<uuid>']`, so both can be queried with cheap
prefix and term lookups.

What this notebook covers:

| | |
|---|---|
| [1. Connect](#1.-Connect) | the sync client and a token check |
| [2. Setup](#2.-Setup) | roles, metadata schemas, the two GroupTypes |
| [3. Pick images](#3.-Pick-images) | six images to work with |
| [4. Build a garment_full](#4.-Build-a-garment_full) | the one-call upsert; skipping the name lookup |
| [5. From the image side](#5.-From-the-image-side) | every (group, role) an image holds |
| [6. One image, several roles](#6.-One-image,-several-roles) | one membership row per (group, image, role) |
| [7. Validation](#7.-Validation) | missing and unknown roles, bad metadata - all rejected |
| [8. A garment_hang_only](#8.-A-garment_hang_only) | the single-role type |
| [9. Image filters](#9.-Image-filters) | by type, by role, exact (role, group) |
| [10. Soft-delete](#10.-Soft-delete) | delete sets deleted_at and queues the OS scrub |
| [11. Put and reuse deleted UUID group.](#11.-Put-and-reuse-deleted-UUID-group.) | a re-PUT to a deleted UUID revives it |

## 1. Connect

The API is grouped by resource: `client.groups.*`, `client.roles.*`, `client.group_types.*`,
`client.images.*`. `DataRoomClientSync().groups.upsert(...)` blocks; on the async
`DataRoomClient` the same call is awaited.


In [1]:
import os

from dataroom_client import DataRoomClientSync, DataRoomError

os.environ["DATAROOM_API_KEY"] = 'YOUR_KEY_HERE'
os.environ["DATAROOM_API_URL"] = 'http://localhost:8000/api/'

client = DataRoomClientSync()
print('connected to', client.api_url)

try:
    client.images.list(limit=1)
except DataRoomError as e:
    raise RuntimeError('API token rejected. Check DATAROOM_API_KEY.') from e
print('token ok')

connected to http://localhost:8000/api/
token ok


## 2. Setup

| GroupType           | Roles                                                                        | Required |
| ------------------- | ---------------------------------------------------------------------------- | -------- |
| `garment_full`      | `on_model_front`, `on_model_side`, `on_model_back`, `full_outfit`, `on_hang` | all      |
| `garment_hang_only` | `on_hang`                                                                    | `on_hang`|

### Roles

In [2]:
ROLES = {
    'on_model_front': 'Frontal shot of the garment on a model.',
    'on_model_side':  'Side profile on a model.',
    'on_model_back':  'Back view on a model.',
    'full_outfit':    'Full-body styling shot.',
    'on_hang':        'Garment on a hanger or mannequin, no model.',
}
for name, description in ROLES.items():
    try:
        client.roles.create(name=name, description=description)
        print(f'created  role {name}')
    except DataRoomError as ex:
        print(ex)
        print(f'skipped  role {name} (already exists)')

Client error '400 Bad Request' for url 'http://localhost:8000/api/roles/'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/400
400
{"name":["role with this name already exists."]}
skipped  role on_model_front (already exists)
Client error '400 Bad Request' for url 'http://localhost:8000/api/roles/'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/400
400
{"name":["role with this name already exists."]}
skipped  role on_model_side (already exists)


Client error '400 Bad Request' for url 'http://localhost:8000/api/roles/'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/400
400
{"name":["role with this name already exists."]}
skipped  role on_model_back (already exists)
Client error '400 Bad Request' for url 'http://localhost:8000/api/roles/'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/400
400
{"name":["role with this name already exists."]}
skipped  role full_outfit (already exists)


Client error '400 Bad Request' for url 'http://localhost:8000/api/roles/'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/400
400
{"name":["role with this name already exists."]}
skipped  role on_hang (already exists)


### Metadata schemas

In [3]:
GARMENT_FULL_SCHEMA = {
    'type': 'object',
    'additionalProperties': False,
    'required': ['brand', 'sku'],
    'properties': {
        'brand':  {'type': 'string', 'minLength': 1, 'maxLength': 64},
        'sku':    {'type': 'string', 'minLength': 1, 'maxLength': 64},
        'season': {'type': 'string', 'maxLength': 32},
    },
}

GARMENT_HANG_ONLY_SCHEMA = {
    'type': 'object',
    'additionalProperties': False,
    'required': ['seller'],
    'properties': {
        'seller':      {'type': 'string', 'minLength': 1, 'maxLength': 128},
        'listing_url': {'type': 'string', 'format': 'uri'},
    },
}

### GroupTypes

In [4]:
GROUP_TYPES = [
    dict(
        name='garment_full',
        description='A garment captured from every angle.',
        metadata_schema=GARMENT_FULL_SCHEMA,
        roles=[
            {'role': 'on_model_front', 'is_required': True},
            {'role': 'on_model_side',  'is_required': True},
            {'role': 'on_model_back',  'is_required': True},
            {'role': 'full_outfit',    'is_required': True},
            {'role': 'on_hang',        'is_required': True},
        ],
    ),
    dict(
        name='garment_hang_only',
        description='Reseller single-shot garment.',
        metadata_schema=GARMENT_HANG_ONLY_SCHEMA,
        roles=[{'role': 'on_hang', 'is_required': True}],
    ),
]

# GroupTypes are immutable: create once, never update. The name is baked into
# the OS encoding of every group of that type and the schema/roles gate their
# validation, so the server has no PUT/PATCH for group-types. A create against
# an existing name 400s, which we treat as a no-op.
for spec in GROUP_TYPES:
    try:
        client.group_types.create(**spec)
        print(f'created  group_type {spec["name"]}')
    except DataRoomError:
        print(f'skipped  group_type {spec["name"]} (already exists; immutable)')


skipped  group_type garment_full (already exists; immutable)


skipped  group_type garment_hang_only (already exists; immutable)


### Verify

In [5]:
print('roles:', [r['name'] for r in client.roles.list()])
for t in client.group_types.list():
    req = [r['role'] for r in t['roles'] if r['is_required']]
    print(f"{t['name']:20s} required_roles={req}  metadata_required={t['metadata_schema'].get('required', [])}")

roles: ['detail_shot', 'empty_room', 'flat_lay', 'full_outfit', 'member', 'on_hang', 'on_model_back', 'on_model_front', 'on_model_side', 'single_image', 'staged_room', 'worn_front']
accessory            required_roles=[]  metadata_required=[]
flux_batch           required_roles=[]  metadata_required=[]
garment              required_roles=[]  metadata_required=[]
garment_full         required_roles=['full_outfit', 'on_hang', 'on_model_back', 'on_model_front', 'on_model_side']  metadata_required=['brand', 'sku']
garment_hang_only    required_roles=['on_hang']  metadata_required=['seller']
garment_pair         required_roles=['on_hang', 'on_model_front']  metadata_required=[]
room_scene           required_roles=['empty_room', 'staged_room']  metadata_required=[]
single_image         required_roles=['single_image']  metadata_required=[]


## 3. Pick images

Need 6 - five for `garment_full` (one per required role), one for `garment_hang_only`.

In [6]:
images = client.images.list(limit=10)
image_ids = [img['id'] for img in images]
print(f'{len(image_ids)} images available; first 6:')
for i in image_ids[:6]:
    print(' ', i)
assert len(image_ids) >= 6, 'need at least 6 images - run import_images first'

10 images available; first 6:
  bench_0_0
  bench_0_1
  bench_0_10
  bench_0_100
  bench_0_101
  bench_0_102


## 4. Build a `garment_full`

`groups.upsert` is one PUT - creates if the name is new, replaces metadata + members otherwise. Required roles missing or unknown role -> 400, no half-state.

In [7]:
full = client.groups.upsert(
    name='zara_01455460',
    type='garment_full',
    description='Floral midi dress.',
    metadata={'brand': 'Zara', 'sku': '01455460', 'season': 'SS24'},
    members=[
        {'image_id': image_ids[0], 'role': 'on_model_front'},
        {'image_id': image_ids[1], 'role': 'on_model_side'},
        {'image_id': image_ids[2], 'role': 'on_model_back'},
        {'image_id': image_ids[3], 'role': 'full_outfit'},
        {'image_id': image_ids[4], 'role': 'on_hang'},
    ],
)
print('upserted', full['id'])
for m in client.groups.members(full['id']):
    print(f"  {m['role']:18s} {m['image_id']}")

upserted 1f9761f9-4d8a-4302-8737-9aa8b6eb3661
  on_hang            bench_0_101
  full_outfit        bench_0_100
  on_model_back      bench_0_10
  on_model_side      bench_0_1
  on_model_front     bench_0_0


### Skipping the name lookup

`groups.upsert` defaults to looking the group up by `name` (one extra GET) so re-runs are idempotent, but looking by group_id makes it faster - 


In [8]:
import uuid

# 1) Re-PUT the same group we just created
existing = client.groups.upsert(
    group_id=full['id'],
    name='zara_01455460',
    type='garment_full',
    metadata={'brand': 'Zara', 'sku': '01455460', 'season': 'AW24'},   # season changed
    members=[
        {'image_id': image_ids[0], 'role': 'on_model_front'},
        {'image_id': image_ids[1], 'role': 'on_model_side'},
        {'image_id': image_ids[2], 'role': 'on_model_back'},
        {'image_id': image_ids[3], 'role': 'full_outfit'},
        {'image_id': image_ids[4], 'role': 'on_hang'},
    ],
)
print('PUT-by-existing-uuid ->', existing['id'], existing['metadata'])

# 2) Create a new group with a client-picked UUID
fresh_id = str(uuid.uuid4())
brand_new = client.groups.upsert(
    group_id=fresh_id,
    name='zara_99999999',
    type='garment_full',
    metadata={'brand': 'Zara', 'sku': '99999999'},
    members=[
        {'image_id': image_ids[0], 'role': 'on_model_front'},
        {'image_id': image_ids[1], 'role': 'on_model_side'},
        {'image_id': image_ids[2], 'role': 'on_model_back'},
        {'image_id': image_ids[3], 'role': 'full_outfit'},
        {'image_id': image_ids[4], 'role': 'on_hang'},
    ],
)
print('PUT-with-fresh-uuid    ->', brand_new['id'], '(pre-picked', fresh_id, ')')

PUT-by-existing-uuid -> 1f9761f9-4d8a-4302-8737-9aa8b6eb3661 {'sku': '01455460', 'brand': 'Zara', 'season': 'AW24'}


PUT-with-fresh-uuid    -> 7c64f7fe-7aff-42e4-8f7c-c1a2c40b0206 (pre-picked 7c64f7fe-7aff-42e4-8f7c-c1a2c40b0206 )


## 5. From the image side

Postgres-side hydrated list - `Membership` joined with `Group`.

In [9]:
for entry in client.images.groups(image_ids[0]):
    g = entry['group']
    print(f"  {g['type']:18s} role={entry['role']:18s} group={g['name']!r} ({g['id']})")

  single_image       role=single_image       group='bench_0_0' (e8ed1fb7-98fb-491b-a6f3-aceb39d02023)
  garment_full       role=on_model_front     group='zara_01455460' (1f9761f9-4d8a-4302-8737-9aa8b6eb3661)
  garment_full       role=on_model_front     group='zara_99999999' (7c64f7fe-7aff-42e4-8f7c-c1a2c40b0206)


## 6. One image, several roles

A `Membership` is unique on `(group, image, role)`, so the same image can play
more than one role in a single group - e.g. one shot that is both the
`on_model_front` and the `full_outfit`. `image_count` still counts distinct
images, and `images.groups` returns one entry per (group, role).

In [10]:
# One image can hold several roles in the same group. Here image_ids[0] is
# both the on_model_front shot AND the full_outfit shot of the product.
multi = client.groups.upsert(
    name='zara_multi_role',
    type='garment_full',
    metadata={'brand': 'Zara', 'sku': 'MULTI01'},
    members=[
        {'image_id': image_ids[0], 'role': 'on_model_front'},
        {'image_id': image_ids[0], 'role': 'full_outfit'},   # same image, 2nd role
        {'image_id': image_ids[1], 'role': 'on_model_side'},
        {'image_id': image_ids[2], 'role': 'on_model_back'},
        {'image_id': image_ids[4], 'role': 'on_hang'},
    ],
)
members = client.groups.members(multi['id'])
print(f"{len(members)} membership rows, image_count={multi['image_count']} (distinct images)")
for m in members:
    print(f"  {m['role']:18s} {m['image_id']}")

# image_ids[0] now resolves to two roles within this single group
roles_here = sorted(
    e['role'] for e in client.images.groups(image_ids[0])
    if e['group']['id'] == multi['id']
)
print('image_ids[0] roles in zara_multi_role:', roles_here)

client.groups.delete(multi['id'])   # tidy up so re-runs stay clean

5 membership rows, image_count=4 (distinct images)
  on_hang            bench_0_101
  on_model_back      bench_0_10
  on_model_side      bench_0_1
  full_outfit        bench_0_0
  on_model_front     bench_0_0
image_ids[0] roles in zara_multi_role: ['full_outfit', 'on_model_front']


## 7. Validation

Every call below should raise - bulk PUT validates `{metadata, [roles]}` in one pass.

### Required role missing

In [11]:
try:
    client.groups.upsert(
        name='zara_01455460',
        type='garment_full',
        metadata={'brand': 'Zara', 'sku': '01455460'},
        members=[
            {'image_id': image_ids[0], 'role': 'on_model_front'},
            {'image_id': image_ids[1], 'role': 'on_model_side'},
            {'image_id': image_ids[2], 'role': 'on_model_back'},
            {'image_id': image_ids[3], 'role': 'full_outfit'},
        ],
    )
except DataRoomError as e:
    print('expected error:', e)

expected error: Client error '400 Bad Request' for url 'http://localhost:8000/api/groups/1f9761f9-4d8a-4302-8737-9aa8b6eb3661/'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/400
400
{"required_roles":"missing required role(s) ['on_hang']"}


### Unknown role

In [12]:
try:
    client.groups.upsert(
        name='zara_01455460',
        type='garment_full',
        metadata={'brand': 'Zara', 'sku': '01455460'},
        members=[{'image_id': image_ids[0], 'role': 'studio'}],
    )
except DataRoomError as e:
    print('expected error:', e)

expected error: Client error '400 Bad Request' for url 'http://localhost:8000/api/groups/1f9761f9-4d8a-4302-8737-9aa8b6eb3661/'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/400
400
{"roles":"unknown role(s) ['studio']; allowed: ['full_outfit', 'on_hang', 'on_model_back', 'on_model_front', 'on_model_side']","required_roles":"missing required role(s) ['full_outfit', 'on_hang', 'on_model_back', 'on_model_front', 'on_model_side']"}


### Bad metadata

In [13]:
try:
    client.groups.upsert(name='ds_no_meta', type='garment_full', metadata={})
except DataRoomError as e:
    print('missing required:', e)

try:
    client.groups.upsert(
        name='ds_extra_meta',
        type='garment_full',
        metadata={'brand': 'X', 'sku': '1', 'colour': 'red'},
    )
except DataRoomError as e:
    print('unknown key:', e)

missing required: Client error '400 Bad Request' for url 'http://localhost:8000/api/groups/b2d7d36a-06a0-44a9-9f1e-1ed45b1706fa/'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/400
400
{"metadata":["'brand' is a required property"]}


unknown key: Client error '400 Bad Request' for url 'http://localhost:8000/api/groups/1fa8186f-f144-4540-86db-e08cd1f06970/'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/400
400
{"metadata":["Additional properties are not allowed ('colour' was unexpected)"]}


## 8. A `garment_hang_only`

Same one-call upsert, single role.

In [14]:
hang = client.groups.upsert(
    name='vinted_xyz_42',
    type='garment_hang_only',
    metadata={'seller': 'xyz', 'listing_url': 'https://www.vinted.com/items/xyz_42'},
    members=[{'image_id': image_ids[5], 'role': 'on_hang'}],
)
print('upserted', hang['id'])

upserted 00b89bcb-2faa-492b-84e9-275560f646ae


## 9. Image filters

Three OS queries - type prefix, role prefix, exact `(role, group)` term.

### By type

In [15]:
hits = client.images.list(group_type='garment_full', limit=20)
print(f'garment_full: {len(hits)} image(s)')
for h in hits[:5]:
    print(' ', h['id'])

garment_full: 5 image(s)
  bench_0_0
  bench_0_1
  bench_0_10
  bench_0_100
  bench_0_101


### By role

In [16]:
hits = client.images.list(roles=['on_hang'], limit=20)
print(f'role=on_hang anywhere: {len(hits)} image(s)')
for h in hits:
    print(' ', h['id'])

role=on_hang anywhere: 1 image(s)
  bench_0_101


### Exact (role, group)

In [17]:
hits = client.images.list(group_ids=[full['id']], roles=['on_model_front'], limit=20)
print(f"on_model_front in {full['name']}: {len(hits)} image(s)")
for h in hits:
    print(' ', h['id'])

on_model_front in zara_01455460: 1 image(s)
  bench_0_0


## 10. Soft-delete

`groups.delete` sets `deleted_at` and queues an OS scrub.

In [18]:
for g in (full, brand_new, hang):
    client.groups.delete(g['id'])
    print('deleted', g['id'])

for t in ('garment_full', 'garment_hang_only'):
    print(f'{t}: {len(client.groups.list(type=t))} remaining')

deleted 1f9761f9-4d8a-4302-8737-9aa8b6eb3661
deleted 7c64f7fe-7aff-42e4-8f7c-c1a2c40b0206


deleted 00b89bcb-2faa-492b-84e9-275560f646ae


garment_full: 0 remaining
garment_hang_only: 0 remaining


## 11. Put and reuse `deleted` UUID group.

A PUT to the UUID of a soft-deleted group clears `deleted_at` and replays the body - same row, same UUID, back in the active set. Useful for retry-safe upserts: the client doesn't need to know whether the group was previously deleted.

In [19]:
# `full` was just soft-deleted. Re-PUTting to its UUID brings it back.
resurrected = client.groups.upsert(
    group_id=full['id'],
    name='zara_01455460',
    type='garment_full',
    metadata={'brand': 'Zara', 'sku': '01455460', 'season': 'SS24'},
    members=[
        {'image_id': image_ids[0], 'role': 'on_model_front'},
        {'image_id': image_ids[1], 'role': 'on_model_side'},
        {'image_id': image_ids[2], 'role': 'on_model_back'},
        {'image_id': image_ids[3], 'role': 'full_outfit'},
        {'image_id': image_ids[4], 'role': 'on_hang'},
    ],
)
print('resurrected', resurrected['id'], '- same uuid:', resurrected['id'] == full['id'])
back = False
for g in client.groups.list(type='garment_full'):
    if g['id'] == resurrected['id']:
        back = True
print('visible in listing again:', back)

# Tidy up so re-runs of the notebook stay clean.
client.groups.delete(resurrected['id'])

resurrected 1f9761f9-4d8a-4302-8737-9aa8b6eb3661 - same uuid: True


visible in listing again: True
